# Boeing 737 MAX and MCAS: When a Single Sensor Can Override a Trained Human

In this notebook I look at the Boeing 737 MAX's MCAS system as a ninth mechanism behind an AI ethics failure, and the first one in this series that isn't really about data, fairness, or consent at all.

Every earlier notebook in this series involved a score, a dataset, a scraped photo, or a live software feedback loop. This one involves an aircraft. MCAS is included here because it's one of the clearest real-world illustrations of what happens when an automated system is given the authority to override a trained human, is built to trust a single, unchecked source of input, and never tells that human it exists in the first place, three failures that show up constantly in software-only AI systems too, just with lower stakes than they had here.

In this notebook, I will:

- Summarize what MCAS was, what it relied on, and what happened on two flights
- Simulate a rare hardware sensor fault feeding into a system that consults only one sensor at a time
- Measure how often that turns a rare fault into a repeated, undiagnosed automated action
- Try the real first fix, a cockpit alert with no other changes, and see how little it moves the outcome
- Find the fix that actually addresses the mechanism: checking a second sensor before acting, and telling the humans the system exists at all

## 1. Background: What Was Reported

The Boeing 737 MAX was built with new, larger, more fuel-efficient engines mounted further forward and higher on the wing than earlier 737 models. That change altered the aircraft's handling: under certain conditions, mainly at low speed and high angle of attack, the new engine placement gave the plane a tendency to pitch its nose up more than pilots trained on earlier 737s would expect. To keep the MAX flying enough like its predecessors that airlines could certify it under the same existing type rating, avoiding costly new pilot simulator training, Boeing added a flight-control software system called MCAS, the Maneuvering Characteristics Augmentation System, that automatically pushed the aircraft's nose down whenever it detected a high angle of attack, without any pilot action.

MCAS took its angle-of-attack reading from a single sensor at a time, even though every 737 MAX has two angle-of-attack sensors, one on each side of the aircraft's nose, with no logic comparing the two readings against each other before acting. If that one sensor failed or reported a bad reading, MCAS could not tell the difference between a real high-angle-of-attack event and a broken sensor, and would repeatedly command the aircraft's nose-down trim, re-triggering every few seconds even against a pilot pulling back on the controls, based entirely on bad data. MCAS itself was not documented in the aircraft's flight manual or pilot training materials, so flight crews had no way of knowing the system existed, let alone recognizing its behavior or the specific procedure to disable it, when it activated.

On October 29, 2018, Lion Air Flight 610 crashed into the Java Sea shortly after takeoff, killing all 189 people on board, after a faulty angle-of-attack sensor triggered repeated, erroneous MCAS nose-down commands the crew could not diagnose in time. On March 10, 2019, Ethiopian Airlines Flight 302 crashed shortly after takeoff in nearly identical circumstances, killing all 157 people on board. Regulators worldwide grounded the entire 737 MAX fleet within days of the second crash, a grounding that lasted about twenty months. A subsequent US House Committee on Transportation and Infrastructure report found that MCAS's authority to move the aircraft's stabilizer had been significantly increased late in the plane's development without a corresponding update to the safety analysis behind it, and that the Federal Aviation Administration had delegated much of the MAX's certification review to Boeing's own engineers, reducing the amount of independent scrutiny the design received before it reached the flying public.

## 2. Why This Is a Different Mechanism

None of the first eight notebooks in this series involved a mechanical, safety-critical system at all, they were about scores, data, consent, and public-facing software behavior. MCAS is a different kind of failure again: a single point of sensor failure, one piece of hardware that could feed bad data to a system with the authority to repeatedly move a passenger aircraft's control surfaces, with no second sensor consulted to catch it.

Layered on top of that is a second, independent problem: the humans who were supposed to be the ultimate fallback if the automation went wrong, the pilots, were never told the automation existed, so when it did go wrong, there was no one in the loop who recognized what was happening quickly enough to intervene. Redundancy and disclosure are not fairness concepts or data concepts, they are among the most basic tools of safety engineering, and this case is what happens when both are missing from a system with the authority to override a trained human.

## 3. A Note on the Simulation Below

This notebook does not use Boeing's or the FAA's real failure-rate figures or system specifications, which involve safety engineering analysis far beyond what a notebook like this should attempt to reproduce. I use small, illustrative probabilities, a made-up sensor fault rate, a made-up pilot recovery rate, to make one mechanism concrete: how relying on a single, unchecked sensor turns a rare hardware fault into a repeated, hard-to-diagnose automated action, and how much that risk shrinks once a second sensor is consulted before the system is allowed to act.

## 4. Simulating a Rare Angle-of-Attack Sensor Fault

I simulate a large number of individual flights. On a small fraction of them, one of the aircraft's two angle-of-attack sensors develops a fault, a bad electrical connection, physical damage, a miscalibration, and reports an incorrect high-angle-of-attack reading for the rest of that flight.

In [ ]:
import random

random.seed(9)

N_FLIGHTS = 200000
SENSOR_FAULT_RATE = 0.0004


def make_flights(n, fault_rate):
    """Returns a list of flight dicts, each labeled with which single AoA sensor, if any, has a fault."""
    flights = []
    for _ in range(n):
        has_fault = random.random() < fault_rate
        faulty_sensor = random.choice(["A", "B"]) if has_fault else None
        flights.append({"faulty_sensor": faulty_sensor})
    return flights


flights = make_flights(N_FLIGHTS, SENSOR_FAULT_RATE)

## 5. Modeling MCAS's Single-Sensor Reliance

The real MCAS took its angle-of-attack reading from whichever flight control computer happened to be active for a given flight, effectively one sensor at a time, with no comparison against the other. I model that by assigning each flight a single active sensor at random, and checking whether that happens to be the one carrying the fault.

In [ ]:
def assign_active_sensor_and_check_trigger(flights):
    """Returns the same flights, each labeled with its active sensor and whether MCAS erroneously triggers."""
    for flight in flights:
        active_sensor = random.choice(["A", "B"])
        flight["active_sensor"] = active_sensor
        flight["erroneous_trigger"] = flight["faulty_sensor"] == active_sensor
    return flights


flights = assign_active_sensor_and_check_trigger(flights)

## 6. Modeling an Undocumented System's Effect on Pilot Recovery

MCAS was undocumented, so on a flight where it triggers erroneously, the crew is dealing with a repeated, unexplained nose-down force with nothing in their manual telling them what it is or the specific, immediate procedure to stop it. I model the odds of the crew correctly diagnosing and disabling it in time as low, reflecting a real system nobody told them existed, and count how many erroneous triggers end without a successful recovery.

In [ ]:
PILOT_RECOVERY_PROB_V1 = 0.15


def simulate_outcomes(flights, recovery_prob, seed):
    """Returns the count of erroneous triggers and the count of those not recovered from in time."""
    rng = random.Random(seed)
    triggered = [f for f in flights if f["erroneous_trigger"]]
    not_recovered = [f for f in triggered if rng.random() >= recovery_prob]
    return len(triggered), len(not_recovered)


triggered_v1, not_recovered_v1 = simulate_outcomes(flights, PILOT_RECOVERY_PROB_V1, seed=1)
print("Erroneous MCAS triggers, undocumented single-sensor design:", triggered_v1)
print("Of those, not recovered from in time:", not_recovered_v1)

## 7. Trying the Real First Fix: A Cockpit Alert Alone

After the first crash, one real response was to make an existing "AoA Disagree" alert, previously an optional, extra-cost feature on many aircraft, a standard cockpit indication. That tells the crew the two sensors disagree, without explaining MCAS, documenting its behavior, or changing what MCAS itself does. I model that as a modest improvement to the crew's odds of recovering in time, reflecting a general warning rather than a specific, trained-for procedure.

In [ ]:
PILOT_RECOVERY_PROB_ALERT_ONLY = 0.35

triggered_alert, not_recovered_alert = simulate_outcomes(flights, PILOT_RECOVERY_PROB_ALERT_ONLY, seed=1)
print("Erroneous MCAS triggers, alert-only fix:", triggered_alert)
print("Of those, not recovered from in time:", not_recovered_alert)

## 8. Why the Alert Alone Isn't Enough

The number of erroneous triggers doesn't move at all, an alert-only fix does nothing about the single-sensor design that produces them in the first place, it only changes the crew's odds after one has already started. And even with the alert, a meaningful share of these events still go unrecovered, because a general warning that something disagrees is not the same as knowing exactly what is happening and exactly what to do about it inside a window of only a few seconds.

## 9. The Fix That Actually Addresses the Mechanism

The fix that actually addresses this doesn't wait for the crew to respond faster, it stops the system from acting on unreliable data in the first place. The redesigned MCAS compares both angle-of-attack sensors before acting and inhibits itself if they disagree beyond a set threshold, and Boeing also limited MCAS to a single, bounded correction instead of one that could re-trigger indefinitely, while making MCAS's existence and behavior part of pilot training and documentation for the first time. I model the sensor cross-check: an erroneous trigger now requires both sensors to be wrong at once, which for an isolated single-sensor fault essentially never happens, and only shows up if a rare, shared cause affects both sensors together.

In [ ]:
def assign_dual_sensor_trigger(flights, common_cause_rate, seed):
    """Returns flights labeled with whether MCAS erroneously triggers once both sensors are cross-checked."""
    rng = random.Random(seed)
    for flight in flights:
        if flight["faulty_sensor"] is None:
            flight["erroneous_trigger_redundant"] = False
        else:
            flight["erroneous_trigger_redundant"] = rng.random() < common_cause_rate
    return flights


COMMON_CAUSE_RATE = 0.01
flights = assign_dual_sensor_trigger(flights, COMMON_CAUSE_RATE, seed=5)

In [ ]:
triggered_redundant = len([f for f in flights if f["erroneous_trigger_redundant"]])

print("Erroneous MCAS triggers, single-sensor design:", triggered_v1)
print("Erroneous MCAS triggers, cross-checked dual-sensor design:", triggered_redundant)

## 10. What Investigators Found Beyond the Sensor Itself

The sensor mechanism is only part of what investigators found. MCAS's authority to move the stabilizer, how far and how repeatedly it could push the nose down, was substantially increased late in the aircraft's development, after early flight testing, without the FAA's safety analysis being updated to reflect the more powerful version, so the version that reached the flying public was more dangerous than the one originally assessed as safe.

## 11. A Certification Process With Less Independent Scrutiny Than It Looked Like

Investigators also found that the FAA had delegated a large share of the MAX's certification work to Boeing's own engineers, under a decades-old program meant to make certification more efficient. That meant much of the review of MCAS, including the late increase in its authority, never received a fully independent look before the plane was approved to fly, a governance gap sitting entirely outside the system itself.